In [1]:
!git clone https://github.com/BeUnMerreHuman/Anime-Character-Re-Identification.git

fatal: destination path 'Anime-Character-Re-Identification' already exists and is not an empty directory.


In [2]:
%cd Anime-Character-Re-Identification

/kaggle/working/Anime-Character-Re-Identification


In [3]:
!git clone https://github.com/Intellindust-AI-Lab/DEIMv2.git

fatal: destination path 'DEIMv2' already exists and is not an empty directory.


In [4]:
%mv deimv2_dinov3_l_anime_finetune.yml DEIMv2/configs/deimv2/deimv2_dinov3_l_anime_finetune.yml
%mv anime_detection.yml DEIMv2/configs/dataset/anime_detection.yml

mv: cannot stat 'deimv2_dinov3_l_anime_finetune.yml': No such file or directory
mv: cannot stat 'anime_detection.yml': No such file or directory


In [5]:
!pip install -q faster-coco-eval
!pip install -q -r DEIMv2/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 8.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 811.5 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 69.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 147.5/363.4 MB 8.7 MB/s eta 0:00:25^C
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 147.5/363.4 MB 8.6 MB/s eta 0:00:26
ERROR: Operation cancelled by user


In [6]:
!kaggle datasets download muneeburrehman98/danbooru-annotated-images

Dataset URL: https://www.kaggle.com/datasets/muneeburrehman98/danbooru-annotated-images
License(s): MIT
danbooru-annotated-images.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -q danbooru-annotated-images.zip

replace __huggingface_repos__.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch

REPO_ID = "Intellindust/DEIMv2_DINOv3_L_COCO"
FILENAME = "model.safetensors"
DEST_DIR = "DEIMv2/ckpts"

token = os.getenv("HF_TOKEN")

print(f"Starting download: {FILENAME} from {REPO_ID}...")

path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir=DEST_DIR,
    token=token
)

print(f"Successfully downloaded to: {path}")

weights = load_file("DEIMv2/ckpts/model.safetensors")
torch.save(
    {"model": weights},
    "DEIMv2/ckpts/deimv2.pth"
)

In [ ]:
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

dataset_dir = "dataset"
seed = 42

dataset_path = Path(dataset_dir)
master_json = dataset_path / "annotations.json"

train_json = dataset_path / "train_annotations.json"
val_json = dataset_path / "val_annotations.json"
test_json = dataset_path / "test_annotations.json"

if train_json.exists() and val_json.exists() and test_json.exists():
    print("Splits already exist")
    exit()

with open(master_json, 'r') as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

buckets = [img.get('bucket', 'unknown') for img in images]

train_imgs, temp_imgs, train_buckets, temp_buckets = train_test_split(
    images, buckets, test_size=0.30, stratify=buckets, random_state=seed
)

val_imgs, test_imgs = train_test_split(
    temp_imgs, test_size=0.50, stratify=temp_buckets, random_state=seed
)

train_ids = {img['id'] for img in train_imgs}
val_ids = {img['id'] for img in val_imgs}
test_ids = {img['id'] for img in test_imgs}

train_data = {
    "images": train_imgs,
    "annotations": [ann for ann in annotations if ann['image_id'] in train_ids],
    "categories": categories
}

val_data = {
    "images": val_imgs,
    "annotations": [ann for ann in annotations if ann['image_id'] in val_ids],
    "categories": categories
}

test_data = {
    "images": test_imgs,
    "annotations": [ann for ann in annotations if ann['image_id'] in test_ids],
    "categories": categories
}

with open(train_json, 'w') as f: json.dump(train_data, f)
with open(val_json, 'w') as f: json.dump(val_data, f)
with open(test_json, 'w') as f: json.dump(test_data, f)

print(f"Train Set: {len(train_imgs)}, Validation Set: {len(val_imgs)}, Test Set: {len(test_imgs)}")

In [ ]:
!python DEIMv2/train.py \
    -c DEIMv2/configs/deimv2/deimv2_dinov3_l_anime_finetune.yml \
    --tuning DEIMv2/ckpts/deimv2.pth \
    --use-amp

In [ ]:
!python DEIMv2/train.py \
  -c DEIMv2/configs/deimv2/deimv2_dinov3_l_anime_finetune.yml \
  -t outputs/deimv2_dinov3_l_anime/best_stg2.pth \
  --test-only \
  -u val_dataloader.dataset.ann_file=dataset/test_annotations.json \
  > outputs/deimv2_dinov3_l_anime/test_results.txt

In [ ]:
import torch
from safetensors.torch import save_file

def convert_to_safetensors(pth_path: str, safetensors_path: str):
    print(f"Loading checkpoint from: {pth_path}")
    
    checkpoint = torch.load(pth_path, map_location="cpu", weights_only=False)
    
    if "ema" in checkpoint and checkpoint["ema"] is not None:
        print("Extracting EMA weights.")
        state_dict = checkpoint["ema"].get("module", checkpoint["ema"])
    elif "model" in checkpoint:
        print("EMA not found. Extracting standard base model weights...")
        state_dict = checkpoint["model"]
    else:
        print("No 'model' or 'ema' key found.")
        state_dict = checkpoint

    clean_state_dict = {}
    seen_pointers = set()
    
    for key, value in state_dict.items():
        if not isinstance(value, torch.Tensor):
            print(f"Skipping non-tensor key: {key}")
            continue
        
        ptr = value.data_ptr()
        if ptr in seen_pointers:
            print(f"Breaking shared memory link for duplicate tensor: {key}")
            clean_state_dict[key] = value.clone().contiguous()
        else:
            seen_pointers.add(ptr)
            clean_state_dict[key] = value.contiguous()

    print(f"Saving stripped, clean weights to: {safetensors_path}")
    save_file(clean_state_dict, safetensors_path)
    print("Conversion complete.")

if __name__ == "__main__":
    INPUT_PTH = "outputs/deimv2_dinov3_l_anime/best_stg2.pth" 
    OUTPUT_SAFETENSORS = "outputs/deimv2_dinov3_l_anime/model.safetensors"
    
    convert_to_safetensors(INPUT_PTH, OUTPUT_SAFETENSORS)

In [ ]:
import json
import os
import sys
from DEIMv2.engine.core import YAMLConfig

def generate_safetensors_config(yaml_config_path, output_json_path):
    
    if not os.path.exists(yaml_config_path):
        raise FileNotFoundError(f"YAML configuration not found at {yaml_config_path}")
    
    config_parser = YAMLConfig(yaml_config_path)
    global_cfg = config_parser.global_cfg 

    safetensors_config = {
        "DEIMTransformer": global_cfg.get("DEIMTransformer", {}),
        "DINOv3STAs": global_cfg.get("DINOv3STAs", {}),
        "HybridEncoder": global_cfg.get("HybridEncoder", {})
    }

    if "PostProcessor" in global_cfg:
        safetensors_config["PostProcessor"] = global_cfg["PostProcessor"]
    else:
        transformer_cfg = safetensors_config.get("DEIMTransformer", {})
        safetensors_config["PostProcessor"] = {
            "num_classes": transformer_cfg.get("num_classes", 1),
            "num_top_queries": transformer_cfg.get("num_queries", 300)
        }

    os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

    with open(output_json_path, 'w') as f:
        json.dump(safetensors_config, f, indent=2, sort_keys=True)

    print(f"Exported model configuration to {output_json_path}")

if __name__ == "__main__":
    
    target_yaml = "DEIMv2/configs/deimv2/deimv2_dinov3_l_anime_finetune.yml"
    target_json = "outputs/deimv2_dinov3_l_anime/config.json"
    
    generate_safetensors_config(target_yaml, target_json)

In [ ]:
import os
import shutil

KEEP_DIR = "outputs"
BASE_DIR = os.getcwd()  

for item in os.listdir(BASE_DIR):
    item_path = os.path.join(BASE_DIR, item)

    if item == KEEP_DIR:
        continue

    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.remove(item_path)
            print(f"Deleted file: {item_path}")
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f"Deleted folder: {item_path}")
    except Exception as e:
        print(f"Failed to delete {item_path}: {e}")